# Nessie & Iceberg Demo - Versioning, Time Travel & Branching

This notebook demonstrates the key features of **Project Nessie** and **Apache Iceberg**:

1. **Git-like Versioning** - Every change is committed and tracked
2. **Time Travel** - Query data as it was at any point in time
3. **Branching** - Create isolated development environments
4. **Merging** - Promote changes from dev to production
5. **Rollback** - Revert to previous states

## Setup - Initialize Spark and Connect to Nessie

In [ ]:
import sys
import os
import requests

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark
from lakehouse.settings import NESSIE_URI

spark = get_spark("nessie-demo")

print("Spark session connected to Nessie catalog")

## Part 1: Explore Nessie Catalog Structure

In [ ]:
# Show all namespaces in Nessie
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

# List all tables across namespaces
print("\n=== Tables in Bronze ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

print("\n=== Tables in Silver ===")
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

print("\n=== Tables in Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

## Part 2: Iceberg Time Travel - Query Historical Snapshots

Iceberg stores **every version** of your data. You can time travel to any snapshot.

In [ ]:
# View the snapshot history of the Bronze sales table
print("=== Snapshot History of Bronze Sales ===")
bronze_history = spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id,
        is_current_ancestor
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""")
bronze_history.show(truncate=False)

# Get snapshot details
print("\n=== Snapshot Details ===")
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation,
        summary['total-records'] as total_records
    FROM nessie.bronze.sales.snapshots
    ORDER BY committed_at
""").show(truncate=False)

### Time Travel: Query a Specific Snapshot

Use `VERSION AS OF` to query data as it was at a specific snapshot.

In [ ]:
# Get the current count
current_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.bronze.sales").first()[0]
print(f"Current Bronze table count: {current_count:,}")

# Get the first snapshot ID
first_snapshot_id = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.history 
    ORDER BY made_current_at 
    LIMIT 1
""").first()[0]

print(f"\nFirst snapshot ID: {first_snapshot_id}")

# Query the first snapshot (time travel!)
print("\n=== Data from First Snapshot ===")
spark.sql(f"""
    SELECT COUNT(*) as count_at_first_snapshot
    FROM nessie.bronze.sales VERSION AS OF '{first_snapshot_id}'
""").show()

### Time Travel: Query by Timestamp

You can also query data as it was at a specific point in time.

In [ ]:
# Get timestamps from history
timestamps = spark.sql("""
    SELECT made_current_at
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").collect()

if len(timestamps) > 1:
    first_timestamp = timestamps[0]['made_current_at']
    print(f"First snapshot timestamp: {first_timestamp}")
    
    # Query using timestamp
    print("\n=== Time Travel by Timestamp ===")
    spark.sql(f"""
        SELECT COUNT(*) as count_at_timestamp
        FROM nessie.bronze.sales TIMESTAMP AS OF '{first_timestamp}'
    """).show()

## Part 3: Compare Snapshots - What Changed?

Compare data between two snapshots to see what was added.

In [ ]:
# Get snapshot IDs for comparison
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at
    FROM nessie.bronze.sales.snapshots
    ORDER BY committed_at
""").collect()

if len(snapshots) >= 2:
    # Get first and current snapshots
    first_snap = snapshots[0]['snapshot_id']
    current_snap = snapshots[-1]['snapshot_id']
    
    # Count records in each snapshot
    first_count = spark.sql(f"""
        SELECT COUNT(*) as cnt 
        FROM nessie.bronze.sales VERSION AS OF '{first_snap}'
    """).first()[0]
    
    current_count = spark.sql(f"""
        SELECT COUNT(*) as cnt 
        FROM nessie.bronze.sales VERSION AS OF '{current_snap}'
    """).first()[0]
    
    print(f"First snapshot records: {first_count:,}")
    print(f"Current snapshot records: {current_count:,}")
    print(f"Records added: {current_count - first_count:,}")
    
    # Find records that are in current but not in first snapshot
    # (This shows the newly ingested data)
    if current_count > first_count:
        print("\n=== Records Added After First Snapshot ===")
        current_data = spark.sql(f"SELECT * FROM nessie.bronze.sales VERSION AS OF '{current_snap}'")
        first_data = spark.sql(f"SELECT * FROM nessie.bronze.sales VERSION AS OF '{first_snap}'")
        
        # Find new records (those with newer ingestion timestamps)
        first_ts = spark.sql(f"""
            SELECT MIN(ingestion_ts) as min_ts
            FROM nessie.bronze.sales VERSION AS OF '{first_snap}'
        """).first()[0]
        
        spark.sql(f"""
            SELECT `Order ID`, `Order Date`, Sales, Profit
            FROM nessie.bronze.sales 
            WHERE ingestion_ts > '{first_ts}'
            LIMIT 5
        """).show(truncate=False)

## Part 4: Nessie Branching - Git-like Data Development

Nessie allows you to create **branches** for isolated development, just like Git!

In [ ]:
# Check current Nessie branch
current_branch = spark.conf.get("spark.sql.catalog.nessie.ref", "main")
print(f"Current Nessie branch: {current_branch}")

### Create a Development Branch

We'll create a `dev` branch to experiment without affecting the main branch.

In [ ]:
# Create a new branch called 'dev-experiment'
branch_name = "dev-experiment"

try:
    response = requests.post(
        f"{NESSIE_URI}/trees",
        json={
            "name": branch_name,
            "type": "BRANCH",
            "sourceRefName": "main"
        }
    )
    if response.status_code in [200, 201]:
        print(f"Branch '{branch_name}' created successfully!")
    elif response.status_code == 409:
        print(f"Branch '{branch_name}' already exists.")
    else:
        print(f"Response: {response.status_code} - {response.text}")
except Exception as e:
    print(f"Note: Make sure Nessie is running: {e}")

In [ ]:
# List all branches in Nessie
try:
    response = requests.get(f"{NESSIE_URI}/trees")
    if response.status_code == 200:
        trees = response.json()
        print("=== Nessie Branches ===")
        for tree in trees.get('trees', []):
            tree_type = tree.get('type', 'UNKNOWN')
            name = tree.get('name', 'unknown')
            hash_id = tree.get('hash', 'N/A')[:8]
            print(f"  [{tree_type}] {name} @ {hash_id}")
except Exception as e:
    print(f"Error: {e}")

### Work on the Dev Branch

Switch Spark to use the dev branch and make changes.

In [ ]:
# Switch Spark to use the dev branch
spark.conf.set("spark.sql.catalog.nessie.ref", "dev-experiment")
print(f"Now working on branch: {spark.conf.get('spark.sql.catalog.nessie.ref')}")

# The dev branch starts with the same data as main
print("\n=== Tables on dev-experiment branch (inherited from main) ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

### Make Experimental Changes on Dev Branch

Let's create a test table that only exists on the dev branch.

In [ ]:
# Create an experimental aggregation on dev branch
# This won't affect the main branch!

# Read silver data
sales_silver = spark.table("nessie.silver.sales")

# Create a new experimental aggregate: sales by ship mode
from pyspark.sql.functions import sum as spark_sum, round as spark_round, count

experimental_gold = sales_silver.groupBy("ship_mode").agg(
    spark_round(spark_sum("sales"), 2).alias("total_sales"),
    spark_round(spark_sum("profit"), 2).alias("total_profit"),
    count("order_id").alias("order_count")
)

print("=== Experimental Aggregation: Sales by Ship Mode ===")
experimental_gold.show(truncate=False)

# Write to a new table on dev branch only
experimental_gold.writeTo("nessie.gold.sales_by_ship_mode").using("iceberg").create()

print("\nTable 'nessie.gold.sales_by_ship_mode' created on dev-experiment branch.")

### Verify Dev Branch Has Different Tables

The dev branch now has a table that doesn't exist on main!

In [ ]:
# Show tables on dev branch
print("=== Tables on dev-experiment branch ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

# Switch back to main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

print("\n=== Tables on main branch (should NOT have sales_by_ship_mode) ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

## Part 5: Merge Dev Branch to Main

Promote the experimental changes from dev to production (main branch).

In [ ]:
# Merge dev-experiment into main using Nessie API
try:
    # Get the hash of the dev branch
    response = requests.get(f"{NESSIE_URI}/trees/{branch_name}")
    if response.status_code == 200:
        dev_hash = response.json().get('hash')
        
        # Merge dev into main
        merge_response = requests.post(
            f"{NESSIE_URI}/trees/main/merge",
            json={
                "fromHash": dev_hash,
                "fromRefName": branch_name
            }
        )
        
        if merge_response.status_code in [200, 201, 204]:
            print(f"Successfully merged '{branch_name}' into main!")
        else:
            print(f"Merge response: {merge_response.status_code} - {merge_response.text}")
    
except Exception as e:
    print(f"Note: {e}")

In [ ]:
# Verify the merged table is now on main
print("=== Tables on main branch after merge ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

# Query the newly merged table
print("\n=== Merged Table Data ===")
spark.sql("SELECT * FROM nessie.gold.sales_by_ship_mode ORDER BY total_sales DESC").show(truncate=False)

## Part 6: Rollback - Revert to a Previous State

One of Nessie's most powerful features: you can rollback to any previous commit!

In [ ]:
# Get the commit log for the main branch
try:
    response = requests.get(f"{NESSIE_URI}/trees/main/history")
    if response.status_code == 200:
        commits = response.json()
        print("=== Recent Commits on Main Branch ===")
        for i, commit in enumerate(commits.get('logEntries', [])[:5]):
            commit_id = commit.get('hash', 'N/A')[:8]
            author = commit.get('author', 'N/A')
            committer = commit.get('committer', 'N/A')
            message = commit.get('message', 'N/A')
            time = commit.get('commitTime', 'N/A')
            print(f"  [{i+1}] {commit_id} by {author}: {message}")
            
        # Save the second-to-last commit hash for rollback demo
        if len(commits.get('logEntries', [])) >= 2:
            previous_hash = commits['logEntries'][1]['hash']
            print(f"\nPrevious commit hash: {previous_hash[:8]}...")
            
except Exception as e:
    print(f"Error: {e}")

### Perform Rollback

Reset main branch to a previous commit (effectively "undoing" changes).

In [ ]:
# For demo purposes, we'll show how rollback works
# Uncomment below to actually perform the rollback

print("=== Rollback Demo ===")
print("To rollback to a previous commit:")
print(f"  requests.assign('{NESSIE_URI}/trees/main', hash='{previous_hash[:8]}...')")
print("\nThis would revert the main branch to the previous state.")
print("\nNote: For this demo, we won't actually rollback to preserve the merged table.")

## Part 7: Summary - Key Takeaways

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║          NESSIE & ICEBERG - KEY FEATURES DEMONSTRATED                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. TIME TRAVEL                                                      ║
║     - Query data as of a specific snapshot ID                        ║
║     - Query data as of a specific timestamp                          ║
║     - Compare snapshots to see what changed                          ║
║                                                                      ║
║  2. BRANCHING (Git-like for Data)                                    ║
║     - Create branches for isolated development                       ║
║     - Make changes on dev branch without affecting main              ║
║     - Merge branches to promote changes to production                ║
║                                                                      ║
║  3. VERSION CONTROL                                                  ║
║     - Every write creates a new commit                               ║
║     - Full history of all changes                                    ║
║     - Rollback to any previous state                                 ║
║                                                                      ║
║  4. ACID TRANSACTIONS                                                ║
║     - Atomic commits (all or nothing)                                ║
║     - Consistent reads across snapshots                              ║
║     - Isolated branches for concurrent development                   ║
║     - Durable storage in S3                                          ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# Switch back to main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")
print("\nDemo complete. Spark session connected to Nessie 'main' branch.")